In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, lil_matrix
from typing import Dict, Tuple, Optional
import itertools


class PUBOToQUBOReducer:
    """
    Редукция PUBO → QUBO через переменные y_ij = x_i * x_j
    
    Для каждого произведения вводится одна вспомогательная переменная
    """
    
    def __init__(self):
        self.n_original = 0
        self.pair_to_aux = {}  # отображение (i,j) -> индекс вспомогательной
        self.aux_to_pair = {}  # отображение индекс -> (i,j)
    
    def reduce(self, H1: np.ndarray, H2: np.ndarray, H3: np.ndarray, penalty: float) -> csr_matrix:
        """
        Редукция PUBO в QUBO
        
        Args:
            H1: (N,) линейные члены
            H2: (N,N) квадратичные
            H3: (N,N,N) кубические
        
        Returns:
            Q: разреженная матрица (M x M)
        """
        self.n_original = len(H1)
        
        # Находим все пары, которые реально нужны
        needed_pairs = set()
        
        c3 = 0
        
        # Из кубических членов
        for i in range(self.n_original):
            for j in range(i+1, self.n_original):
                for k in range(j+1, self.n_original):
                    if abs(H3[i, j, k]) > 1e-10:
                        c3 += 1
                        needed_pairs.add((i, j))
                        
        print("количество троек", c3)
        # Сортируем пары для детерминированного порядка
        needed_pairs = sorted(needed_pairs)
        
        # Создаем отображения
        self.pair_to_aux = {pair: idx + self.n_original 
                           for idx, pair in enumerate(needed_pairs)}
        self.aux_to_pair = {idx + self.n_original: pair 
                           for idx, pair in enumerate(needed_pairs)}
        
        n_aux = len(needed_pairs)
        n_qubo = self.n_original + n_aux
        
        print(f"Исходных переменных: {self.n_original}")
        print(f"Вспомогательных (пары): {n_aux}")
        print(f"Всего QUBO переменных: {n_qubo}")
        
        # Строим разреженную матрицу
        Q = lil_matrix((n_qubo, n_qubo), dtype=np.float64)
        b = np.zeros(n_qubo)
        
        # 1. Линейные  квадратичные члены 
        for i in range(self.n_original):
            b[i] = H1[i]
            for j in range(i+1, self.n_original):
                Q[i, j] += H2[i, j] / 2
                Q[j, i] += H2[i, j] / 2
                
                    
        for pair in self.pair_to_aux:
            
            aux = self.pair_to_aux[pair]
            i, j = pair
            
            # penalty * (x_0*x_1 - 2x_0*y - 2x_1*y - x_0 - x_1 + 2y)
            b[i] += -1 * penalty  
            b[j] += -1 * penalty  
            b[aux] += 2 * penalty  
            Q[i, j] += penalty / 2          
            Q[j, i] += penalty / 2         
            Q[i, aux] += -2 * penalty / 2  
            Q[aux, i] += -2 * penalty / 2  
            Q[j, aux] += -2 * penalty / 2     
            Q[aux, j] += -2 * penalty / 2  
        
        

        
        # 3. Кубические члены x_i * x_j * x_k = y_ij * x_k
        for i in range(self.n_original):
            for j in range(i+1, self.n_original):
                for k in range(j+1, self.n_original):
                    if abs(H3[i, j, k]) > 1e-10:
                        pair_ij = (i, j)
                        aux = self.pair_to_aux[pair_ij]
                        Q[aux, k] += 2 * H3[i, j, k] / 2
                        Q[k, aux] += 2 * H3[i, j, k] / 2
                        
                        Q[i, k] += -1 * H3[i, j, k] / 2
                        Q[k, i] += -1 * H3[i, j, k] / 2
                        Q[j, k] += -1 * H3[i, j, k] / 2
                        Q[k, j] += -1 * H3[i, j, k] / 2
                        
                        b[k] += H3[i, j, k]
                        
        
        return Q.tocsr(), b, self.pair_to_aux
    
    def recover_solution(self, qubo_sol: np.ndarray) -> np.ndarray:
        """
        Восстанавливает исходное решение
        
        Args:
            qubo_sol: решение QUBO задачи
        
        Returns:
            решение для исходных переменных
        """
        orig = qubo_sol[:self.n_original].copy()
        
        # Проверяем согласованность со вспомогательными
        for aux, pair in self.aux_to_pair.items():
            if aux < len(qubo_sol):
                i, j = pair
                y_ij = qubo_sol[aux]
                prod = orig[i] * orig[j]
                
                # Если не совпадает - корректируем (выбираем большинство)
                if abs(y_ij - prod) > 0.5:
                    if y_ij == 1:
                        # y_ij=1 значит i и j должны быть 1
                        orig[i] = 1
                        orig[j] = 1
        
        return orig

In [ ]:

def example():
    """Пример"""
    N = 10
    H1 = np.random.randn(N) * 0.1
    H2 = np.random.randn(N, N) * 0.01
    H3 = np.random.randn(N, N, N) * 0.001
    
    # Делаем разреженными
    H2[np.abs(H2) < 0.05] = 0
    H3[np.abs(H3) < 0.01] = 0
    
    reducer = PUBOToQUBOReducer()
    Q = reducer.reduce(H1, H2, H3)
    
    print(f"\nQUBO матрица:")
    print(f"  Форма: {Q.shape}")
    print(f"  Ненулевых: {Q.nnz}")
    print(f"  Плотность: {Q.nnz / (Q.shape[0]**2) * 100:.4f}%")


if __name__ == "__main__":
    example()